# VonDoom: Digital Lock System
**Swayam Shakya**

A runnable software lock and a close look at its architecture. Run cells in order. The normal walkthrough uses only the standard library and disposable databases. The final two sections are opt-in.

The public sequence circuit and private-PIN controller are separate models. The circuit explains digital logic; the controller demonstrates local software authorization. Neither operates physical hardware.


## 1. Load the shared implementation
Start this notebook from the extracted project folder.

In [1]:
from pathlib import Path
import sys
import math
import tempfile
import time
import unittest

ROOT = Path.cwd()
if not (ROOT / 'locksystem').is_dir():
    raise RuntimeError('Open Jupyter from the VonDoom folder, then restart the kernel.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from locksystem import DigitalLock, SequenceLock, gate_next, table_next
from locksystem.logic import decode
print('Shared Python modules loaded.')

Shared Python modules loaded.


## 2. Decode four input bits
For key 10, the input is 1010. For key 6 it is 0110; for key 9 it is 1001. A decoder outputs one only when all four literals match.

$$I_{10}=B_3\overline{B_2}B_1\overline{B_0},\quad I_6=\overline{B_3}B_2B_1\overline{B_0},\quad I_9=B_3\overline{B_2}\overline{B_1}B_0$$


In [2]:
print('Key  Binary  I10 I6 I9')
for key in range(16):
    a, b, c = decode(key)
    print(f'{key:>3}  {key:04b}     {a}  {b}  {c}')

Key  Binary  I10 I6 I9
  0  0000     0  0  0
  1  0001     0  0  0
  2  0010     0  0  0
  3  0011     0  0  0
  4  0100     0  0  0
  5  0101     0  0  0
  6  0110     0  1  0
  7  0111     0  0  0
  8  1000     0  0  0
  9  1001     0  0  1
 10  1010     1  0  0
 11  1011     0  0  0
 12  1100     0  0  0
 13  1101     0  0  0
 14  1110     0  0  0
 15  1111     0  0  0


## 3. Derive and verify the state logic
S0=00, S1=01, S2=10, S3=11. D1 is high only on S1+key6 or S2+key9. D0 is high only on S0+key10 or S2+key9.

$$D_1=\overline{Q_1}Q_0 I_6 + Q_1\overline{Q_0}I_9$$
$$D_0=\overline{Q_1}\overline{Q_0}I_{10} + Q_1\overline{Q_0}I_9$$
$$G=Q_1Q_0,\quad R=\overline{G}$$

The reference model uses the transition specification. The gate implementation uses bit operations. Comparing them for every pair catches wrong state qualifications and decoder inversions.


In [3]:
checked = 0
for state in range(4):
    for key in range(16):
        assert gate_next(state, key) == table_next(state, key), (state, key)
        checked += 1
print(f'{checked}/64 transitions agree.')

64/64 transitions agree.


## 4. Clock the sequence detector
Each call is one clock edge. The printed output is the state after the edge. Wrong input resets progress; idle holds state.

In [4]:
circuit = SequenceLock()
print('Edge Key Old New Green Red')
for edge, key in enumerate([10, 6, 9, 0, 10, 7, 10, 6, 9], 1):
    old = circuit.state
    new = circuit.press(key)
    print(f'{edge:>4} {key:>3} {old:02b}  {new:02b}    {int(circuit.unlocked)}    {int(not circuit.unlocked)}')
circuit.reset()
assert circuit.state == 0

Edge Key Old New Green Red
   1  10 00  01    0    1
   2   6 01  10    0    1
   3   9 10  11    1    0
   4   0 11  00    0    1
   5  10 00  01    0    1
   6   7 01  00    0    1
   7  10 00  01    0    1
   8   6 01  10    0    1
   9   9 10  11    1    0


## 5. Run the PIN controller
This example PIN is public test data, not a default credential. The real web console asks you to choose your own PIN. The temporary database is removed when this cell ends.


In [5]:
with tempfile.TemporaryDirectory() as folder:
    lock = DigitalLock(Path(folder) / 'demo.sqlite3')
    try:
        print('Before setup:', lock.status()['state'])
        lock.setup('483729')
        print('Wrong PIN accepted:', lock.unlock('999999')['ok'])
        print('Correct PIN accepted:', lock.unlock('483729')['ok'])
        print(lock.protected_content()['message'])
        lock.lock()
        print('After manual lock:', lock.status()['state'])
        try:
            lock.protected_content()
        except PermissionError:
            print('Protected response correctly denied.')
    finally:
        lock.close()

Before setup: UNCONFIGURED
Wrong PIN accepted: False
Correct PIN accepted: True
Access granted. This response passed the Python lock's authorization check.
After manual lock: LOCKED
Protected response correctly denied.


## 6. Test time without waiting
A clock is injected for reproducible tests. Production uses wall time for persistent cooldowns and monotonic time for the open window. No test needs to wait half a minute.


In [6]:
class TestClock:
    def __init__(self):
        self.value = 1000.0
    def __call__(self):
        return self.value
    def advance(self, seconds):
        self.value += seconds

clock = TestClock()
lock = DigitalLock(':memory:', wall_clock=clock, clock=clock)
try:
    lock.setup('483729')
    for _ in range(5):
        lock.unlock('999999')
    assert lock.status()['state'] == 'LOCKOUT'
    assert not lock.unlock('483729')['ok']
    print('Five failed attempts: LOCKOUT; even the correct PIN is temporarily denied.')
    clock.advance(30)
    assert lock.unlock('483729')['ok']
    print('At 30 seconds: correct PIN opens the lock.')
    clock.advance(15)
    assert lock.status()['state'] == 'LOCKED'
    print('After another 15 seconds: automatically LOCKED.')
finally:
    lock.close()

Five failed attempts: LOCKOUT; even the correct PIN is temporarily denied.
At 30 seconds: correct PIN opens the lock.
After another 15 seconds: automatically LOCKED.


### Check a wall-clock change
An active cooldown now uses monotonic time, just like the open window. Changing wall time must not bypass it while this process is running. Restart recovery still uses the saved wall deadline.


In [7]:
wall, monotonic = TestClock(), TestClock()
with DigitalLock(':memory:', wall_clock=wall, clock=monotonic) as lock:
    lock.setup('483729')
    for _ in range(5):
        lock.unlock('999999')
    wall.advance(3600)
    assert not lock.unlock('483729')['ok']
    print('Jumping wall time forward one hour did not bypass the active cooldown.')
    monotonic.advance(30)
    assert lock.unlock('483729')['ok']
    print('Thirty monotonic seconds later, the correct PIN works.')

Jumping wall time forward one hour did not bypass the active cooldown.
Thirty monotonic seconds later, the correct PIN works.


### Keep one owner per database
Two controllers would otherwise have separate timers and could disagree about the same lock. An operating-system file lock prevents the second controller from opening the database.


In [8]:
with tempfile.TemporaryDirectory() as folder:
    database = Path(folder) / 'owned.sqlite3'
    with DigitalLock(database):
        try:
            DigitalLock(database)
        except OSError:
            print('Second controller rejected while the first owns the database.')
        else:
            raise AssertionError('Database ownership was not enforced.')
    with DigitalLock(database) as lock:
        print('Ownership released cleanly:', lock.status()['state'])

Second controller rejected while the first owns the database.
Ownership released cleanly: UNCONFIGURED


## 7. Computational mathematics
The published sequence has no secrecy. Entropy calculations below describe uniformly random choices only. A short, human-chosen PIN usually provides less protection than its theoretical maximum.


In [9]:
state_bits = math.ceil(math.log2(4))
sequence_space = 16 ** 3
pin_space = 10 ** 6
print('State bits:', state_bits)
print('Random sequence space:', sequence_space)
print('Random sequence entropy:', math.log2(sequence_space), 'bits')
print(f'Uniform six-digit PIN entropy: {math.log2(pin_space):.2f} bits')
rate_per_minute = 5 * 60 / 30
print('Simplified sustained online ceiling:', rate_per_minute, 'guesses/minute')
print('Approximate mean search time:', pin_space / 2 / rate_per_minute, 'minutes')

State bits: 2
Random sequence space: 4096
Random sequence entropy: 12.0 bits
Uniform six-digit PIN entropy: 19.93 bits
Simplified sustained online ceiling: 10.0 guesses/minute
Approximate mean search time: 50000.0 minutes


## 8. Measure gate evaluation
This benchmark measures Python execution, not physical gate propagation. Results depend on the machine and interpreter. A benchmark is not a security guarantee.


In [10]:
iterations = 100_000
start = time.perf_counter()
state = 0
for i in range(iterations):
    state = gate_next(state, (10, 6, 9)[i % 3])
elapsed = time.perf_counter() - start
print(f'{iterations:,} gate evaluations in {elapsed:.4f} seconds')
print(f'Mean evaluation time: {elapsed / iterations * 1e6:.2f} microseconds')

100,000 gate evaluations in 0.1082 seconds
Mean evaluation time: 1.08 microseconds


## 9. Run the complete tests
Tests cover the controller, HTTP boundary, persistence, concurrency, timing and all 64 transitions. Most behavioral tests use a reduced hash cost; one smoke test checks the production cost.

In [11]:
suite = unittest.defaultTestLoader.discover(str(ROOT / 'tests'))
result = unittest.TextTestRunner(verbosity=1, stream=sys.stdout).run(suite)
assert result.wasSuccessful(), 'Review test failures before continuing.' 

......................................................
----------------------------------------------------------------------
Ran 54 tests in 2.740s

OK


## 10. Optional private-PIN exercise
Set the flag to True and rerun this cell to type a private PIN into a hidden prompt. It stays in a temporary demonstration database. Do not use a PIN you use elsewhere. Do not save private data in notebook outputs.


In [12]:
RUN_PRIVATE_DEMO = False
if RUN_PRIVATE_DEMO:
    from getpass import getpass
    with tempfile.TemporaryDirectory() as folder:
        private = DigitalLock(Path(folder) / 'private.sqlite3')
        try:
            private.setup(getpass('Choose 6–12 digits: '))
            result = private.unlock(getpass('Enter the same PIN: '))
            print('Access granted.' if result['ok'] else 'Access denied.')
        finally:
            private.close()
else:
    print('Private exercise skipped. Set RUN_PRIVATE_DEMO = True to try it.')

Private exercise skipped. Set RUN_PRIVATE_DEMO = True to try it.


## 11. Launch the web console from this notebook
Set RUN_WEB to True. This starts the same server as `python server.py`. Open the printed URL in a browser on this computer. The kernel must keep running. Stop any standalone instance first. This server uses the real project database, not the temporary databases above.


In [13]:
RUN_WEB = False
if RUN_WEB:
    from server import start_background, stop_background
    if 'web_server' not in globals() or web_server is None:
        web_server = start_background(port=8765)
    print(f'Open http://127.0.0.1:{web_server.server_port}')
else:
    print('Set RUN_WEB = True to start the local web console.')

Set RUN_WEB = True to start the local web console.


## 12. Stop the notebook-hosted server
Run this cell after you finish using the web console. It is deliberately gated so Run All does not immediately stop your chosen server.

In [14]:
STOP_WEB = False
if STOP_WEB and globals().get('web_server') is not None:
    stop_background(web_server)
    web_server = None
    print('Web controller stopped.')
else:
    print('Set STOP_WEB = True when you want to stop the notebook server.')

Set STOP_WEB = True when you want to stop the notebook server.


## Continue exploring
- Read `docs/ENGINEERING.md` for the derivations, register model, HTTP interface and threat boundary.
- Inspect `docs/truth-table.csv` for all 64 rows.
- Open `index.html` for the offline circuit or use the local server for the PIN console.
- Resetting the sequence circuit never changes the private PIN controller.

The local OS account is trusted. This project gates a demonstration response; it is not an encrypted vault or a physical access device.
